# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TobyRathmell123/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd, numpy as np, os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages loaded")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
30000 pages loaded


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# My rule: a page is worth reviewing first if it's stale (hasn't been updated in a long time) AND is still visible (people are actually searching and landing on it). A stale page nobody sees isn't worth fixing as no one would notice the improvement. A visible page that was just updated isn't urgent either, someone already touched it recently.
# Reason code: stale_visible_page: the only code my rule outputs.
# Action: refresh for flagged pages, monitor for everything else.

# Signal 1: staleness -> decline rate
# This is the assumption behind FlyRank's real refresh flags: "old page = declining page"
staleness_check = (
    df.groupby("freshness_tier")["is_declining_label"]
    .agg(decline_rate="mean", n="count")
)
print("Signal 1: freshness_tier vs decline rate")
print(staleness_check.round(3))
#This data confirms that as freshness tier increases, decline rate generally increases apart from above 180, showing mixed results.

# Signal 2: CTR vs position — the assumption behind FlyRank's CTR-fix logic
# Volume floor (impressions >= 100) so a rate from 3 clicks doesn't fool us
visible = df[df["impressions_90d"] >= 100]
ctr_check = (
    visible.groupby("position_tier")["ctr"]
    .agg(mean_ctr="mean", n="count")
    .sort_values("mean_ctr", ascending=False)
)
print("Signal 2: position_tier vs mean CTR (impressions >= 100)")
print(ctr_check.round(4))

#This shows confirmed results taht as position increases, ctr decreases.

Signal 1: freshness_tier vs decline rate
                decline_rate      n
freshness_tier                     
0-30                   0.511  20480
181+                   0.471    174
31-90                  0.589    175
91-180                 0.611   9171
Signal 2: position_tier vs mean CTR (impressions >= 100)
               mean_ctr     n
position_tier                
page_1           0.3548  8633
top_3            0.3341   533
striking         0.2558  5903
page_3_5         0.1424  6058
deep             0.0554   879


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = np.where((stale == 1) & (visible == 1), "stale_visible_page", "")
df["action"] = np.where((stale == 1) & (visible == 1), "refresh", "monitor")

df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)
queue = df.sort_values("baseline_rank")

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "reason_code", "action", "is_declining_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
]
queue[output_columns].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")

Wrote 30000 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)[
    ["baseline_rank", "action", "reason_code", "is_declining_label",
     "impressions_90d", "days_since_last_update", "avg_position"]
]
print(top10.to_string(index=False))
# Rank 1: action=refresh, reason=stale_visible_page. Flagged because 61678 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 2: action=refresh, reason=stale_visible_page. Flagged because 59472 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 3: action=refresh, reason=stale_visible_page. Flagged because 25715 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 4: action=refresh, reason=stale_visible_page. Flagged because 13299 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 5: action=refresh, reason=stale_visible_page. Flagged because 7812 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 6: action=refresh, reason=stale_visible_page. Flagged because 7558 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 7: action=refresh, reason=stale_visible_page. Flagged because 4590 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 8: action=refresh, reason=stale_visible_page. Flagged because 4556 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 9: action=refresh, reason=stale_visible_page. Flagged because 4429 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.
# Rank 10: action=refresh, reason=stale_visible_page. Flagged because 1697 impressions and untouched for 190+ days. Actually declining: yes. Would be wrong if this page had just recovered, my rule can't see recent trend, only current traffic and update age.


 baseline_rank  action        reason_code  is_declining_label  impressions_90d  days_since_last_update  avg_position
             1 refresh stale_visible_page                   1            61678                     194          19.7
             2 refresh stale_visible_page                   1            59472                     194          24.8
             3 refresh stale_visible_page                   1            25715                     194          22.2
             4 refresh stale_visible_page                   1            13299                     193          10.5
             5 refresh stale_visible_page                   1             7812                     194          39.0
             6 refresh stale_visible_page                   1             7558                     193          17.9
             7 refresh stale_visible_page                   1             4590                     194          31.0
             8 refresh stale_visible_page                   1   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
misses = top10[top10["is_declining_label"] == 0]
print(f"{len(misses)} of your top 10 are NOT actually declining")
print(misses)

0 of your top 10 are NOT actually declining
Empty DataFrame
Columns: [baseline_rank, action, reason_code, is_declining_label, impressions_90d, days_since_last_update, avg_position]
Index: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.